# How to test exceptions and error handling

A practical guide to testing that your code properly raises exceptions for invalid inputs and error conditions.

## Prerequisites

This guide assumes you:
- Understand basic Python exceptions
- Have completed [Your First Test](../learn/01-your-first-test.ipynb)
- Know how to run tests in Jupyter notebooks

## What you'll learn

By the end of this guide, you'll know how to:
- Test that functions raise the correct exceptions
- Validate exception messages and attributes
- Choose between different testing approaches
- Test real-world error scenarios
- Avoid common testing mistakes

## Why test exceptions?

Testing exceptions is crucial because:

- **Ensuring error handling works**: Your defensive code should catch and report errors properly
- **Validating user feedback**: Error messages should be clear and actionable
- **Preventing silent failures**: Code should fail loudly with clear exceptions, not silently return wrong results
- **Documenting expected behaviour**: Tests show what inputs are invalid and why

**Example scenario**: A function that calculates age from birthdate should raise `ValueError` for dates in the future, not return a negative age.

## Basic exception testing

Let's start with a simple example: testing division by zero.

In [ ]:
def divide(a, b):
    """Divide a by b."""
    if b == 0:
        raise ZeroDivisionError("Cannot divide by zero")
    return a / b

### Using assertRaises with callable

The callable approach passes the function and its arguments to `assertRaises()`:

In [ ]:
import unittest

class TestDivideCallable(unittest.TestCase):
    """Test divide using callable approach."""
    
    def test_divide_by_zero_raises_error(self):
        """Test that dividing by zero raises ZeroDivisionError."""
        self.assertRaises(ZeroDivisionError, divide, 10, 0)
        
# Run the test
unittest.main(argv=[''], verbosity=2, exit=False)

**How it works:**
- `assertRaises(exception_type, callable, *args, **kwargs)`
- First argument: exception type to expect
- Second argument: function to call
- Remaining arguments: passed to the function
- Test passes if the function raises the expected exception

**Pros:**
- Concise for simple cases
- Clear what function is being tested

**Cons:**
- Can't easily inspect the exception
- Awkward for complex function calls

## Context manager approach (recommended)

The context manager approach uses `with` to create a block where you expect an exception:

In [ ]:
class TestDivideContextManager(unittest.TestCase):
    """Test divide using context manager approach."""
    
    def test_divide_by_zero_raises_error(self):
        """Test that dividing by zero raises ZeroDivisionError."""
        with self.assertRaises(ZeroDivisionError):
            divide(10, 0)
            
# Run the test
unittest.main(argv=[''], verbosity=2, exit=False)

**Advantages of context manager approach:**

1. **More readable**: Looks like regular code
2. **Flexible**: Can test complex expressions or multiple lines
3. **Access exception**: Can inspect exception details

**This is the recommended approach for most cases.**

## Validating exception details

Often, you want to check not just that an exception was raised, but also that it has the correct message or attributes.

### Checking exception messages

Use the context manager with the `as` clause to capture the exception:

In [ ]:
class TestExceptionMessages(unittest.TestCase):
    """Test exception message content."""
    
    def test_divide_by_zero_has_correct_message(self):
        """Test that division by zero has a clear error message."""
        with self.assertRaises(ZeroDivisionError) as cm:
            divide(10, 0)
        
        # Check the exception message
        self.assertEqual(str(cm.exception), "Cannot divide by zero")
        
# Run the test
unittest.main(argv=[''], verbosity=2, exit=False)

**How it works:**
- `as cm` captures the exception context
- `cm.exception` gives you the actual exception object
- `str(cm.exception)` gets the exception message

**Why validate messages?**
- Ensures error messages are helpful to users
- Catches accidental message changes
- Documents expected error descriptions

### Using assertRaisesRegex for pattern matching

When the exact message might vary but should contain certain text, use `assertRaisesRegex()`:

In [ ]:
def validate_age(age):
    """Validate that age is reasonable."""
    if age < 0:
        raise ValueError(f"Age must be positive, got {age}")
    if age > 150:
        raise ValueError(f"Age must be less than 150, got {age}")
    return age

class TestAgeValidationWithRegex(unittest.TestCase):
    """Test age validation with regex patterns."""
    
    def test_negative_age_error_message(self):
        """Test that negative age error mentions 'positive'."""
        with self.assertRaisesRegex(ValueError, r'positive'):
            validate_age(-5)
    
    def test_excessive_age_error_mentions_limit(self):
        """Test that excessive age error mentions the limit."""
        with self.assertRaisesRegex(ValueError, r'150'):
            validate_age(200)
            
# Run the test
unittest.main(argv=[''], verbosity=2, exit=False)

**When to use assertRaisesRegex:**
- Error message includes variable values (like the actual input)
- You care about message content but not exact wording
- You want to check for key terms without being brittle

**Regex tips:**
- Use raw strings (`r'...'`) for regex patterns
- Keep patterns simple and focused
- Match key terms, not entire messages

## Testing multiple exception types

Functions often raise different exceptions for different error conditions.

In [ ]:
def calculate_discount(price, percentage):
    """Calculate discounted price."""
    if not isinstance(price, (int, float)):
        raise TypeError(f"Price must be a number, got {type(price).__name__}")
    if price < 0:
        raise ValueError("Price cannot be negative")
    if percentage < 0 or percentage > 100:
        raise ValueError("Percentage must be between 0 and 100")
    return price * (1 - percentage / 100)

class TestCalculateDiscountExceptions(unittest.TestCase):
    """Test various exception scenarios for calculate_discount."""
    
    def test_non_numeric_price_raises_type_error(self):
        """Test that non-numeric price raises TypeError."""
        with self.assertRaises(TypeError):
            calculate_discount("100", 10)
    
    def test_negative_price_raises_value_error(self):
        """Test that negative price raises ValueError."""
        with self.assertRaises(ValueError):
            calculate_discount(-10, 10)
    
    def test_negative_percentage_raises_value_error(self):
        """Test that negative percentage raises ValueError."""
        with self.assertRaises(ValueError):
            calculate_discount(100, -5)
    
    def test_excessive_percentage_raises_value_error(self):
        """Test that percentage over 100 raises ValueError."""
        with self.assertRaises(ValueError):
            calculate_discount(100, 150)
            
# Run the tests
unittest.main(argv=[''], verbosity=2, exit=False)

**Key principles:**
- Use `TypeError` for wrong type (not a number, not a string, etc.)
- Use `ValueError` for wrong value (negative when positive expected, out of range, etc.)
- One test per exception scenario for clarity
- Descriptive test names explain what condition triggers the exception

## Real-world scenarios

Let's look at common situations where exception testing is essential.

### Scenario 1: File operations

Testing that code properly handles missing files:

In [ ]:
import json

def load_config(filename):
    """Load configuration from JSON file."""
    try:
        with open(filename, 'r') as f:
            return json.load(f)
    except FileNotFoundError:
        raise FileNotFoundError(f"Configuration file not found: {filename}")
    except json.JSONDecodeError as e:
        raise ValueError(f"Invalid JSON in {filename}: {e}")

class TestLoadConfig(unittest.TestCase):
    """Test configuration file loading."""
    
    def test_missing_file_raises_file_not_found_error(self):
        """Test that missing config file raises FileNotFoundError."""
        with self.assertRaises(FileNotFoundError) as cm:
            load_config('nonexistent.json')
        
        # Check that error message mentions the filename
        self.assertIn('nonexistent.json', str(cm.exception))
            
# Run the test
unittest.main(argv=[''], verbosity=2, exit=False)

**Why this matters:**
- File operations commonly fail in production
- Users need clear error messages about what went wrong
- Tests ensure error handling doesn't silently fail

### Scenario 2: API input validation

Testing that an API endpoint properly validates user input:

In [ ]:
def create_user(username, email, age):
    """Create a new user account."""
    if not username or len(username) < 3:
        raise ValueError("Username must be at least 3 characters")
    
    if '@' not in email or '.' not in email:
        raise ValueError(f"Invalid email address: {email}")
    
    if not isinstance(age, int):
        raise TypeError("Age must be an integer")
    
    if age < 13:
        raise ValueError("User must be at least 13 years old")
    
    return {'username': username, 'email': email, 'age': age}

class TestCreateUser(unittest.TestCase):
    """Test user creation validation."""
    
    def test_short_username_raises_value_error(self):
        """Test that username under 3 characters is rejected."""
        with self.assertRaisesRegex(ValueError, r'at least 3 characters'):
            create_user('ab', 'test@example.com', 25)
    
    def test_invalid_email_raises_value_error(self):
        """Test that invalid email format is rejected."""
        with self.assertRaises(ValueError) as cm:
            create_user('alice', 'not-an-email', 25)
        
        self.assertIn('Invalid email', str(cm.exception))
    
    def test_non_integer_age_raises_type_error(self):
        """Test that non-integer age is rejected."""
        with self.assertRaises(TypeError):
            create_user('alice', 'alice@example.com', '25')
    
    def test_underage_user_raises_value_error(self):
        """Test that users under 13 are rejected."""
        with self.assertRaisesRegex(ValueError, r'at least 13'):
            create_user('alice', 'alice@example.com', 12)
            
# Run the tests
unittest.main(argv=[''], verbosity=2, exit=False)

**Why comprehensive validation testing matters:**
- Prevents invalid data from entering your system
- Ensures helpful error messages guide users to fix problems
- Documents validation rules in executable form

### Scenario 3: Custom exceptions

Testing domain-specific exceptions:

In [ ]:
class InsufficientFundsError(Exception):
    """Raised when account has insufficient funds."""
    pass

class Account:
    """Simple bank account."""
    
    def __init__(self, balance=0):
        self.balance = balance
    
    def withdraw(self, amount):
        """Withdraw money from account."""
        if amount > self.balance:
            raise InsufficientFundsError(
                f"Cannot withdraw £{amount}, balance is £{self.balance}"
            )
        self.balance -= amount
        return self.balance

class TestAccountWithdrawal(unittest.TestCase):
    """Test account withdrawal with custom exception."""
    
    def test_overdraft_raises_insufficient_funds_error(self):
        """Test that withdrawing more than balance raises InsufficientFundsError."""
        account = Account(balance=100)
        
        with self.assertRaises(InsufficientFundsError) as cm:
            account.withdraw(150)
        
        # Check that error message includes relevant amounts
        error_msg = str(cm.exception)
        self.assertIn('150', error_msg)  # Withdrawal amount
        self.assertIn('100', error_msg)  # Current balance
            
# Run the test
unittest.main(argv=[''], verbosity=2, exit=False)

**Custom exceptions are valuable because:**
- They make error handling more specific and intentional
- They communicate domain concepts (insufficient funds, not found, unauthorized, etc.)
- They allow callers to catch specific errors without catching unrelated ones

**Testing custom exceptions ensures:**
- They're raised in the right circumstances
- They contain useful information for handling the error

## Anti-patterns to avoid

Common mistakes when testing exceptions.

### ❌ Don't use try/except in tests

**Bad approach:**

In [ ]:
class BadExceptionTest(unittest.TestCase):
    """Example of what NOT to do."""
    
    def test_division_by_zero_bad(self):
        """BAD: Using try/except in a test."""
        try:
            divide(10, 0)
            self.fail("Expected ZeroDivisionError")  # If we get here, test should fail
        except ZeroDivisionError:
            pass  # Expected - test passes

**Why this is bad:**
- Verbose and harder to read
- Easy to accidentally catch wrong exceptions
- Doesn't validate exception details
- `assertRaises` does this better

**Good approach:**

In [ ]:
class GoodExceptionTest(unittest.TestCase):
    """Example of the right way."""
    
    def test_division_by_zero_good(self):
        """GOOD: Using assertRaises context manager."""
        with self.assertRaises(ZeroDivisionError):
            divide(10, 0)

### ❌ Don't make tests too broad

**Bad: Catching any exception**

In [ ]:
class TooeBroadTest(unittest.TestCase):
    """Example of overly broad test."""
    
    def test_invalid_input_raises_something(self):
        """BAD: Too vague about what exception is expected."""
        with self.assertRaises(Exception):  # Too broad!
            calculate_discount("invalid", 10)

**Why this is bad:**
- `Exception` catches *any* exception, including unexpected ones
- Won't catch if the wrong exception is raised
- Doesn't document what specific error is expected

**Good: Be specific**

In [ ]:
class SpecificTest(unittest.TestCase):
    """Example of specific, clear test."""
    
    def test_non_numeric_price_raises_type_error(self):
        """GOOD: Specific about expected exception type."""
        with self.assertRaises(TypeError):
            calculate_discount("invalid", 10)

### ❌ Don't forget to actually trigger the exception

**Bad: Test that doesn't fail**

In [ ]:
class IncompleteTest(unittest.TestCase):
    """Example of a test that passes for the wrong reason."""
    
    def test_exception_but_never_raised(self):
        """BAD: assertRaises block is empty."""
        with self.assertRaises(ValueError):
            # Forgot to actually call the function!
            pass  # This test passes, but checks nothing!

**This test passes**, but it's not testing anything! Always ensure the code inside `assertRaises` actually triggers the exception.

## Quick reference

### Basic exception test

```python
def test_something_raises_error(self):
    with self.assertRaises(ValueError):
        function_that_should_fail()
```

### Check exception message

```python
def test_error_has_correct_message(self):
    with self.assertRaises(ValueError) as cm:
        function_that_should_fail()
    
    self.assertEqual(str(cm.exception), "Expected message")
```

### Match message pattern

```python
def test_error_message_contains_keyword(self):
    with self.assertRaisesRegex(ValueError, r'keyword'):
        function_that_should_fail()
```

### Common exception types

- `ValueError`: Wrong value (negative when positive expected, out of range, etc.)
- `TypeError`: Wrong type (string when number expected, etc.)
- `ZeroDivisionError`: Division by zero
- `FileNotFoundError`: File doesn't exist
- `KeyError`: Dictionary key doesn't exist
- `IndexError`: List index out of range
- Custom exceptions: Domain-specific errors

## Summary

Testing exceptions is essential for robust code. Remember:

1. **Use context manager approach** (`with assertRaises`) for flexibility and readability
2. **Validate exception messages** to ensure user feedback is helpful
3. **Be specific** about exception types (use `TypeError`, `ValueError`, not `Exception`)
4. **Test each error condition separately** for clear, focused tests
5. **Use `assertRaisesRegex`** when messages include variable content
6. **Avoid try/except** in tests – use `assertRaises` instead
7. **Test real-world scenarios** like file operations and API validation

## Next steps

- Practice with the [Calculator Example](../examples/basic/) which includes exception tests
- Review [Assertions Reference](../reference/assertions.md) for all exception testing methods
- Learn about [Test Fixtures](../learn/04-test-fixtures.ipynb) for setting up test scenarios
- Read [Avoid Common Testing Mistakes](avoid-common-testing-mistakes.md) for more anti-patterns

Happy testing!